# 🚧 Estruturação de Dados — DNIT × PRF (Brasil — Geral)

**Notebook único de limpeza e estruturação** dos dados do **DNIT** (condição das rodovias, ICM)
e da **PRF** (acidentes), deixando-os prontos para **análise exploratória** e **treinamento de modelos**.

> **Escopo desta entrega:** *estruturar* os dados. A análise exploratória, os modelos preditivos e o
> deploy (Streamlit) serão construídos depois, sobre as bases geradas aqui.

**Regras (pedidos do Athayde):**
1. Padrão de colunas fixo e ordenado.
2. Abrangência: **Brasil inteiro (todas as UFs)** → apenas as **TOP 10 BRs com mais linhas**.
3. Foco em **ter dados** (maximizar volume).
4. Eliminar colunas inúteis (latitude, longitude, observação, mês…).
5. PRF restrita às **mesmas TOP 10 BRs** e ao **mesmo período do DNIT**.
6. Um único notebook, no repositório **PredicaoAcidentesPRF**.

> ⚠️ **Escala do ICM a confirmar:** há divergência entre fontes (maior=melhor × maior=pior).
> Os valores de ICM são mantidos **como no original**; a interpretação fica para a etapa de análise.


## ⚙️ Etapa 0 — Configuração

In [1]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)

# Caminhos
RAW = Path("../data/raw")
PROC = Path("../data/processed")
PROC.mkdir(parents=True, exist_ok=True)

DNIT_CONSOLIDADO = RAW / "dnit_consolidado.csv"   # base do DNIT (já consolidada)
KAGGLE_DATASET = "lucasandroliveira/dnit-prf-norte"  # fonte do DNIT no Kaggle (fallback)

def get_dnit_consolidado():
    # Caminho do dnit_consolidado.csv: usa o local se existir; senão baixa do Kaggle.
    if DNIT_CONSOLIDADO.exists():
        print("DNIT: usando arquivo local ->", DNIT_CONSOLIDADO)
        return DNIT_CONSOLIDADO
    import kagglehub
    print("DNIT: baixando do Kaggle ->", KAGGLE_DATASET)
    base_kaggle = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    # procura recursivamente (resiliente a subpastas na estrutura do dataset)
    achados = list(base_kaggle.rglob("dnit_consolidado.csv"))
    if not achados:
        raise FileNotFoundError("dnit_consolidado.csv não encontrado no dataset Kaggle")
    return achados[0]

# Escopo geográfico e temporal
UFS = None   # None = Brasil inteiro (sem filtro de UF)
ANOS = [2024, 2025, 2026]            # janela de cruzamento DNIT × PRF
N_TOP_BRS = 10

# PRF — Dados Abertos (datatran / agrupados por pessoa). Baixados via gdown se faltarem.
PRF_DRIVE_IDS = {
    2024: "14lVfqdoE2gxDliaKZu7K9Mx6847maPtl",
    2025: "1-Gp9S-ALO0D1nT8S_OKoC8xlW7BY8F82",
    2026: "1B_rvx1kPHuFowP5rs84ZO-57gN8yUs85",
}

def num_br(serie):
    """Extrai o número da BR (ex.: 'BR-364' -> '364', 364.0 -> '364')."""
    return (serie.astype(str).str.extract(r"(\d{1,3})")[0]
            .str.zfill(3))

print("Configuração pronta. Escopo: BRASIL (todas as UFs) | Anos:", ANOS)


Configuração pronta. Escopo: BRASIL (todas as UFs) | Anos: [2024, 2025, 2026]


## 🛣️ Etapa 1 — DNIT (condição das rodovias)

A base `dnit_consolidado.csv` já reúne os levantamentos do DNIT. Aqui apenas **estruturamos**:
escolhemos as TOP 10 BRs com mais linhas (nível Brasil), recortamos a janela de anos,
removemos colunas inúteis e deduplicamos cada trecho pela avaliação **mais recente**.

In [2]:
dnit = pd.read_csv(get_dnit_consolidado(), sep=";", low_memory=False)
print("DNIT consolidado bruto:", dnit.shape)

dnit["uf"] = dnit["uf"].astype(str).str.upper().str.strip()
dnit["br"] = num_br(dnit["rodovia"])
dnit["ano"] = pd.to_numeric(dnit["ano"], errors="coerce")

# Filtro Norte + janela de anos
dnit = dnit[dnit["ano"].isin(ANOS)]   # Brasil inteiro: sem filtro de UF
dnit = dnit.dropna(subset=["br", "km"])

# TOP 10 BRs com MAIS LINHAS (pedido #2)
TOP_BRS = dnit["br"].value_counts().head(N_TOP_BRS).index.tolist()
print("TOP 10 BRs Brasil (por nº de linhas):", TOP_BRS)
dnit = dnit[dnit["br"].isin(TOP_BRS)]


DNIT: usando arquivo local -> ..\data\raw\dnit_consolidado.csv


DNIT consolidado bruto: (3439793, 25)
TOP 10 BRs Brasil (por nº de linhas): ['364', '230', '116', '101', '158', '153', '316', '174', '242', '010']


In [3]:
# Colunas inúteis a remover (pedido #4): coordenadas, observação, mês (já há data/ano)
DROP_DNIT = ["latitude", "longitude", "observacao", "mes", "rodovia",
             "id_malha", "contrato", "icm_unificado", "_arquivo_origem"]
dnit = dnit.drop(columns=[c for c in DROP_DNIT if c in dnit.columns])

# Ordem padrão das colunas (pedido #1)
ORDEM_DNIT = ["uf", "br", "km", "km_inicial", "km_final", "extensao", "sentido",
              "ano", "data_aval", "superficie", "num_faixas",
              "cond_pavimento", "cond_conservacao", "cond_pista",
              "icc", "icp", "icm"]
dnit = dnit[[c for c in ORDEM_DNIT if c in dnit.columns]]

# Dedup: um registro por trecho (uf, br, km), mantendo a avaliação mais recente
dnit["_dt"] = pd.to_datetime(dnit.get("data_aval"), errors="coerce", dayfirst=True)
dnit = (dnit.sort_values(["ano", "_dt"])
            .drop_duplicates(subset=["uf", "br", "km"], keep="last")
            .drop(columns="_dt")
            .reset_index(drop=True))

print("DNIT estruturado:", dnit.shape)
print("Cobertura ICM:", f"{dnit['icm'].notna().mean()*100:.1f}%")
display(dnit.head())


DNIT estruturado: (44318, 17)
Cobertura ICM: 96.1%


,uf,br,km,km_inicial,km_final,extensao,sentido,ano,data_aval,superficie,num_faixas,cond_pavimento,cond_conservacao,cond_pista,icc,icp,icm
0,RS,116,276.50,277.0,276.0,1.0,NaN,2024.0,2024-02-17 00:00:00,NaN,NaN,NaN,X,NaN,20.0,5.0,9.50
1,PE,116,91.30,91.6,91.0,0.6,NaN,2024.0,2024-02-25 00:00:00,NaN,NaN,NaN,X,NaN,25.0,32.5,30.25
2,PE,316,218.65,218.3,219.0,0.7,NaN,2024.0,2024-02-25 00:00:00,NaN,NaN,NaN,X,NaN,25.0,25.0,25.00
3,PE,316,369.25,369.5,369.0,0.5,NaN,2024.0,2024-02-25 00:00:00,NaN,NaN,NaN,X,NaN,25.0,25.0,25.00
4,PE,316,218.15,218.3,218.0,0.3,NaN,2024.0,2024-02-25 00:00:00,NaN,NaN,NaN,X,NaN,25.0,25.0,25.00


In [4]:
saida_dnit = PROC / "dnit_estruturado_brasil.csv"
dnit.to_csv(saida_dnit, index=False, sep=";", encoding="utf-8-sig")
print("Salvo:", saida_dnit.resolve(), f"({saida_dnit.stat().st_size/1e6:.1f} MB)")


Salvo: C:\UFRA\Projetos - Gihub\PredicaoAcidentesPRF\data\processed\dnit_estruturado_brasil.csv (3.5 MB)


## 🚑 Etapa 2 — PRF (acidentes)

Usamos os arquivos **nacionais** dos Dados Abertos da PRF (formato *datatran* / agrupados por pessoa),
2024–2026. Filtramos para as **mesmas TOP 10 BRs** (nível Brasil) do DNIT.

In [5]:
def baixar_prf(ano, fid):
    destino = RAW / f"acidentes{ano}.csv"
    if destino.exists():
        return destino
    import gdown, zipfile
    tmp = RAW / f"prf_{ano}.zip"
    gdown.download(id=fid, output=str(tmp), quiet=False)
    with zipfile.ZipFile(tmp) as zf:
        zf.extractall(RAW)
    tmp.unlink(missing_ok=True)
    return destino

def ler_prf(caminho):
    for sep in (";", ","):
        try:
            d = pd.read_csv(caminho, sep=sep, encoding="latin-1", low_memory=False)
            if d.shape[1] > 5:
                return d
        except Exception:
            pass
    raise RuntimeError(f"Falha ao ler {caminho}")

partes = []
for ano, fid in PRF_DRIVE_IDS.items():
    caminho = baixar_prf(ano, fid)
    partes.append(ler_prf(caminho))
prf = pd.concat(partes, ignore_index=True)
print("PRF nacional (2024-2026):", prf.shape)


PRF nacional (2024-2026): (1436475, 52)


In [6]:
prf["uf"] = prf["uf"].astype(str).str.upper().str.strip()
prf["br"] = num_br(prf["br"])

# Mesmo recorte do DNIT: Norte + TOP 10 BRs
prf = prf[prf["br"].isin(TOP_BRS)].copy()   # Brasil inteiro: filtra só pelas TOP BRs

# Colunas inúteis para o objetivo (coordenadas + administrativas)
DROP_PRF = ["latitude", "longitude", "regional", "delegacia", "uop"]
prf = prf.drop(columns=[c for c in DROP_PRF if c in prf.columns])

# Tipos
prf["km"] = pd.to_numeric(prf["km"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
prf["data_inversa"] = pd.to_datetime(prf["data_inversa"], errors="coerce", format="%Y-%m-%d")
prf["ano"] = prf["data_inversa"].dt.year

prf = prf.dropna(subset=["br", "km"]).reset_index(drop=True)
print("PRF estruturada:", prf.shape)
print("Por ano:", prf["ano"].value_counts().sort_index().to_dict())
print("Por BR:", prf["br"].value_counts().to_dict())
display(prf.head())


PRF estruturada: (524043, 48)
Por ano: {2024: 267898, 2025: 256145}
Por BR: {'116': 163055, '101': 152010, '364': 55339, '153': 47573, '230': 35426, '316': 25580, '158': 13580, '242': 11638, '010': 11027, '174': 8815}


,id,pesid,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_principal,causa_acidente,ordem_tipo_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,id_veiculo,tipo_veiculo,marca,ano_fabricacao_veiculo,tipo_envolvido,estado_fisico,idade,sexo,ilesos,feridos_leves,feridos_graves,mortos,cd_bat,id_envolvido,uf_acidente,rodovia,cond_meteorologica,estrutura_viaria,local_urbanizado,ano_fabricacao,qtde_ileso,qtde_lesoes_leves,qtde_lesoes_graves,qtde_mortos,Sigla da Superintendência,Sigla da Delegacia,Sigla da Unidade Operacional,ano
0,571772.0,1268971.0,2024-01-01,segunda-feira,00:05:00,RJ,101,272.5,TANGUA,Sim,Reação tardia ou ineficiente do condutor,1,Colisão com objeto,Com Vítimas Fatais,Plena Noite,Decrescente,Céu Claro,Dupla,Reta,Sim,1018215.0,Motocicleta,HONDA/CG 160 FAN,2023.0,Condutor,Óbito,26.0,Masculino,0.0,0.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024
1,571774.0,1268985.0,2024-01-01,segunda-feira,00:05:00,GO,153,424.6,ANAPOLIS,Não,Animais na Pista,1,Colisão com objeto,Sem Vítimas,Plena Noite,Decrescente,Céu Claro,Dupla,Reta,Não,1018226.0,Automóvel,VW/GOL 1.6 POWER,2009.0,Condutor,Ileso,34.0,Feminino,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024
2,571774.0,1268985.0,2024-01-01,segunda-feira,00:05:00,GO,153,424.6,ANAPOLIS,Sim,Velocidade Incompatível,1,Colisão com objeto,Sem Vítimas,Plena Noite,Decrescente,Céu Claro,Dupla,Reta,Não,1018226.0,Automóvel,VW/GOL 1.6 POWER,2009.0,Condutor,Ileso,34.0,Feminino,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024
3,571777.0,1269020.0,2024-01-01,segunda-feira,01:45:00,ES,101,264.1,SERRA,Sim,Reação tardia ou ineficiente do condutor,1,Colisão com objeto,Sem Vítimas,Plena Noite,Decrescente,Nublado,Múltipla,Interseção de Vias;Reta,Sim,1018251.0,Caminhonete,VW/SAVEIRO 1.6,2009.0,Condutor,Ileso,55.0,Masculino,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024
4,571778.0,1269028.0,2024-01-01,segunda-feira,00:45:00,SC,101,110.0,PENHA,Não,Reação tardia ou ineficiente do condutor,1,Saída de leito carroçável,Com Vítimas Feridas,Plena Noite,Crescente,Chuva,Dupla,Curva,Não,1018261.0,Camioneta,I/SSANGYONG ACTYON A23 4,2011.0,Condutor,Ileso,53.0,Masculino,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024


In [7]:
saida_prf = PROC / "prf_estruturado_brasil.csv"
prf.to_csv(saida_prf, index=False, sep=";", encoding="utf-8-sig")
print("Salvo:", saida_prf.resolve(), f"({saida_prf.stat().st_size/1e6:.1f} MB)")


Salvo: C:\UFRA\Projetos - Gihub\PredicaoAcidentesPRF\data\processed\prf_estruturado_brasil.csv (171.4 MB)


## 🔗 Etapa 3 — Base cruzada (DNIT × PRF) para modelagem

Cada acidente recebe o ICM do trecho correspondente. **Chave: `uf + br + km arredondado`**
(o DNIT já está deduplicado pela avaliação mais recente de cada trecho).

In [8]:
dnit_key = dnit.copy()
dnit_key["km_join"] = dnit_key["km"].round().astype("Int64")
dnit_key = (dnit_key.drop_duplicates(subset=["uf", "br", "km_join"], keep="last")
                    [["uf", "br", "km_join", "icc", "icp", "icm",
                      "cond_pavimento", "cond_conservacao", "cond_pista"]])

prf_key = prf.copy()
prf_key["km_join"] = prf_key["km"].round().astype("Int64")

base = prf_key.merge(dnit_key, on=["uf", "br", "km_join"], how="left", suffixes=("", "_dnit"))
cobertura = base["icm"].notna().mean() * 100
print("Base cruzada:", base.shape, f"| acidentes com ICM casado: {cobertura:.1f}%")

saida_base = PROC / "base_modelagem_brasil.csv"
base.to_csv(saida_base, index=False, sep=";", encoding="utf-8-sig")
print("Salvo:", saida_base.resolve(), f"({saida_base.stat().st_size/1e6:.1f} MB)")
display(base.head())


Base cruzada: (524043, 55) | acidentes com ICM casado: 52.0%
Salvo: C:\UFRA\Projetos - Gihub\PredicaoAcidentesPRF\data\processed\base_modelagem_brasil.csv (179.5 MB)


,id,pesid,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_principal,causa_acidente,ordem_tipo_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,id_veiculo,tipo_veiculo,marca,ano_fabricacao_veiculo,tipo_envolvido,estado_fisico,idade,sexo,ilesos,feridos_leves,feridos_graves,mortos,cd_bat,id_envolvido,uf_acidente,rodovia,cond_meteorologica,estrutura_viaria,local_urbanizado,ano_fabricacao,qtde_ileso,qtde_lesoes_leves,qtde_lesoes_graves,qtde_mortos,Sigla da Superintendência,Sigla da Delegacia,Sigla da Unidade Operacional,ano,km_join,icc,icp,icm,cond_pavimento,cond_conservacao,cond_pista
0,571772.0,1268971.0,2024-01-01,segunda-feira,00:05:00,RJ,101,272.5,TANGUA,Sim,Reação tardia ou ineficiente do condutor,1,Colisão com objeto,Com Vítimas Fatais,Plena Noite,Decrescente,Céu Claro,Dupla,Reta,Sim,1018215.0,Motocicleta,HONDA/CG 160 FAN,2023.0,Condutor,Óbito,26.0,Masculino,0.0,0.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,272,NaN,NaN,NaN,NaN,NaN,NaN
1,571774.0,1268985.0,2024-01-01,segunda-feira,00:05:00,GO,153,424.6,ANAPOLIS,Não,Animais na Pista,1,Colisão com objeto,Sem Vítimas,Plena Noite,Decrescente,Céu Claro,Dupla,Reta,Não,1018226.0,Automóvel,VW/GOL 1.6 POWER,2009.0,Condutor,Ileso,34.0,Feminino,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,425,NaN,NaN,NaN,NaN,NaN,NaN
2,571774.0,1268985.0,2024-01-01,segunda-feira,00:05:00,GO,153,424.6,ANAPOLIS,Sim,Velocidade Incompatível,1,Colisão com objeto,Sem Vítimas,Plena Noite,Decrescente,Céu Claro,Dupla,Reta,Não,1018226.0,Automóvel,VW/GOL 1.6 POWER,2009.0,Condutor,Ileso,34.0,Feminino,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,425,NaN,NaN,NaN,NaN,NaN,NaN
3,571777.0,1269020.0,2024-01-01,segunda-feira,01:45:00,ES,101,264.1,SERRA,Sim,Reação tardia ou ineficiente do condutor,1,Colisão com objeto,Sem Vítimas,Plena Noite,Decrescente,Nublado,Múltipla,Interseção de Vias;Reta,Sim,1018251.0,Caminhonete,VW/SAVEIRO 1.6,2009.0,Condutor,Ileso,55.0,Masculino,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,264,NaN,NaN,NaN,NaN,NaN,NaN
4,571778.0,1269028.0,2024-01-01,segunda-feira,00:45:00,SC,101,110.0,PENHA,Não,Reação tardia ou ineficiente do condutor,1,Saída de leito carroçável,Com Vítimas Feridas,Plena Noite,Crescente,Chuva,Dupla,Curva,Não,1018261.0,Camioneta,I/SSANGYONG ACTYON A23 4,2011.0,Condutor,Ileso,53.0,Masculino,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,110,NaN,NaN,NaN,NaN,NaN,NaN


## ✅ Saídas e próximos passos

Geradas em `data/processed/`:
- **`dnit_estruturado_brasil.csv`** — DNIT do Brasil (TOP 10 BRs), 1 linha por trecho, avaliação mais recente.
- **`prf_estruturado_brasil.csv`** — acidentes PRF do Brasil nas mesmas BRs (2024–2026).
- **`base_modelagem_brasil.csv`** — acidentes + ICM do trecho (pronto para EDA e modelos).

**A cargo do Athayde (próximas etapas):**
1. Análise exploratória sobre as bases acima.
2. Definição do alvo (ex.: gravidade/feridos/mortos) e *feature engineering* adicional.
3. Treinamento e avaliação dos modelos preditivos.
4. Deploy via **Streamlit**.

> ⚠️ Antes de interpretar o ICM no modelo, **confirmar a direção da escala** (maior = melhor ou pior).
